In [21]:
from pathlib import Path
import pandas as pd

flights = pd.read_csv(Path("Data") / "flights_clean.csv")

# 预测到达延误：保留正常执飞、未备降，而且有 0/1 标签的航班
delay = flights[
    (flights["Cancelled"] == 0)
    & (flights["Diverted"] == 0)
    & flights["ArrDel15"].isin([0, 1])
].copy()

In [22]:
from sklearn.preprocessing import OneHotEncoder

features = ["Reporting_Airline", "Dest", "Month", "DayOfWeek", "DepHour"]
train = delay[delay["Year"].between(2022, 2024)]
val = delay[delay["Year"] == 2025]

# 只从训练集学习有哪些类别，验证集沿用同一套编码
encoder = OneHotEncoder(handle_unknown="ignore")
X_train = encoder.fit_transform(train[features])
X_val = encoder.transform(val[features])

y_train = train["ArrDel15"].astype(int)
y_val = val["ArrDel15"].astype(int)

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for weight in [None, "balanced"]:
    model = LogisticRegression(class_weight=weight, max_iter=500)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    print("class_weight =", weight)
    print("Accuracy:", round(accuracy_score(y_val, pred), 3))
    print("Precision:", round(precision_score(y_val, pred, zero_division=0), 3))
    print("Recall:", round(recall_score(y_val, pred), 3))
    print("F1:", round(f1_score(y_val, pred), 3))

class_weight = None
Accuracy: 0.758
Precision: 0.49
Recall: 0.026
F1: 0.048
class_weight = balanced
Accuracy: 0.625
Precision: 0.34
Recall: 0.588
F1: 0.431


In [24]:
delay_sample = pd.read_csv(Path("Data") / "delay_model_sample.csv")
weather = pd.read_csv(Path("Data") / "booking_weather_features.csv")

# 先只用 BOS 出发地、1999–2021 年这个历史窗口
weather_cols = [
    "Origin_1999_Wind20",
    "Origin_1999_VisBelow3",
    "Origin_1999_CeilingBelow1000",
    "Origin_1999_Snow",
    "Origin_1999_Thunder",
]

# 保留所有航班，用 FlightRowID 找到每班航班对应的天气
data = delay_sample.merge(
    weather[["FlightRowID"] + weather_cols],
    on="FlightRowID",
    how="left",
    validate="one_to_one",
)

# 看每个天气特征有多大比例成功匹配，1.0 代表 100%
print(data[weather_cols].notna().mean().round(3))

Origin_1999_Wind20              1.0
Origin_1999_VisBelow3           1.0
Origin_1999_CeilingBelow1000    1.0
Origin_1999_Snow                1.0
Origin_1999_Thunder             1.0
dtype: float64


In [25]:
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler

train = data[data["Year"].between(2022, 2024)]
val = data[data["Year"] == 2025]

y_train = train["ArrDel15"].astype(int)
y_val = val["ArrDel15"].astype(int)

# A：沿用上个实验的五个时刻表特征和 OneHotEncoder
X_train = encoder.fit_transform(train[features])
X_val = encoder.transform(val[features])

# 天气缺失值用训练集的中位数补。缺失和“发生比例为 0”含义不同
weather_median = train[weather_cols].median()
train_weather = train[weather_cols].fillna(weather_median)
val_weather = val[weather_cols].fillna(weather_median)

# 五种天气的发生频率差异较大，用训练集的均值和标准差做标准化
scaler = StandardScaler()
train_weather = scaler.fit_transform(train_weather)
val_weather = scaler.transform(val_weather)

# B：在 A 的右边加上五列天气，每一行仍然是原来那班航班
X_train_weather = hstack([X_train, train_weather], format="csr")
X_val_weather = hstack([X_val, val_weather], format="csr")

In [30]:
experiments = [
    ("Schedule only", X_train, X_val),
    ("Schedule + BOS climate", X_train_weather, X_val_weather),
    ("Schedule + BOS climate + 6h TAF", X_train_6h, X_val_6h),
]

results = []
for name, train_x, val_x in experiments:
    model = LogisticRegression(class_weight="balanced", max_iter=500)
    model.fit(train_x, y_train)
    pred = model.predict(val_x)

    results.append({
        "Features": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
    })

pd.DataFrame(results).round(4)

,Features,Accuracy,Precision,Recall,F1
0,Schedule only,0.6245,0.3399,0.5884,0.4309
1,Schedule + BOS climate,0.6314,0.3446,0.5834,0.4333
2,Schedule + BOS climate + 6h TAF,0.6344,0.3492,0.5943,0.4399


In [27]:
taf_cols = ["TAFWindKt_6h", "TAFVisBelow3_6h"]

taf = pd.read_csv(
    Path("Data") / "bos_taf_6h_features.csv",
    usecols=["FlightRowID"] + taf_cols,
).set_index("FlightRowID", verify_integrity=True)

# 按照现有训练集和验证集的航班顺序取预报，确保每行对应同一班飞机
train_taf = taf.reindex(train["FlightRowID"])[taf_cols]
val_taf = taf.reindex(val["FlightRowID"])[taf_cols]

# 每一列有多少比例的航班拿到了预报
pd.DataFrame({
    "Train coverage": train_taf.notna().mean(),
    "Validation coverage": val_taf.notna().mean(),
}).round(3)

C:\Users\Owner\AppData\Local\Temp\ipykernel_27336\511923456.py:6: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  ).set_index("FlightRowID", verify_integrity=True)


,Train coverage,Validation coverage
TAFWindKt_6h,0.999,0.999
TAFVisBelow3_6h,0.999,0.999


In [28]:
# 保留同一批航班，用训练集的中位数补缺失值
taf_median = train_taf.median()
taf_scaler = StandardScaler()

train_taf_scaled = taf_scaler.fit_transform(train_taf.fillna(taf_median))
val_taf_scaled = taf_scaler.transform(val_taf.fillna(taf_median))

# X_train_weather 已经包含时刻表 + 五个历史天气特征
X_train_6h = hstack([X_train_weather, train_taf_scaled], format="csr")
X_val_6h = hstack([X_val_weather, val_taf_scaled], format="csr")

In [31]:
from sklearn.tree import DecisionTreeClassifier

tree_results = []

# 使用同一批航班和同一套特征，只比较树的最大深度
for depth in [3, 5, 8]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42,
    )
    tree.fit(X_train_6h, y_train)
    pred = tree.predict(X_val_6h)

    tree_results.append({
        "Max depth": depth,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
    })

pd.DataFrame(tree_results).round(4)

,Max depth,Accuracy,Precision,Recall,F1
0,3,0.6508,0.3323,0.4418,0.3793
1,5,0.6550,0.3392,0.4517,0.3875
2,8,0.6741,0.3543,0.4248,0.3864


In [32]:
# data 已经包含航班和历史天气，再按 FlightRowID 接上两列 TAF
final_data = data.join(taf[taf_cols], on="FlightRowID", validate="one_to_one")

final_train = final_data[final_data["Year"].between(2022, 2025)].copy()
test = final_data[final_data["Year"] == 2026].copy()

print("Training flights:", len(final_train))
print("Test flights:", len(test))
print("Test dates:", test["FlightDate"].min(), "to", test["FlightDate"].max())

Training flights: 550318
Test flights: 80808
Test dates: 2026-01-01 to 2026-07-31


In [33]:
# 时刻表特征：2026 年只使用训练时学到的编码
final_encoder = OneHotEncoder(handle_unknown="ignore")
schedule_train = final_encoder.fit_transform(final_train[features])
schedule_test = final_encoder.transform(test[features])

# 历史天气和 TAF 都是数值列，可以放在一起逐列处理
numeric_cols = weather_cols + taf_cols
medians = final_train[numeric_cols].median()

final_scaler = StandardScaler()
weather_train = final_scaler.fit_transform(final_train[numeric_cols].fillna(medians))
weather_test = final_scaler.transform(test[numeric_cols].fillna(medians))

X_final_train = hstack([schedule_train, weather_train], format="csr")
X_test = hstack([schedule_test, weather_test], format="csr")

In [34]:
from sklearn.metrics import confusion_matrix

final_model = LogisticRegression(class_weight="balanced", max_iter=500)
final_model.fit(X_final_train, final_train["ArrDel15"].astype(int))

y_test = test["ArrDel15"].astype(int)
test_pred = final_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))

pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=[0, 1]),
    index=["Actual: no delay ≥15 min", "Actual: delay ≥15 min"],
    columns=["Predicted: no delay ≥15 min", "Predicted: delay ≥15 min"],
)

Accuracy: 0.6281
Precision: 0.3541
Recall: 0.6386
F1: 0.4556


,Predicted: no delay ≥15 min,Predicted: delay ≥15 min
Actual: no delay ≥15 min,38178,22937
Actual: delay ≥15 min,7118,12575


In [35]:
cancel = pd.read_csv(Path("Data") / "cancel_model_sample.csv")

# 沿用相同的时间划分，先看训练集和验证集
cancel_train = cancel[cancel["Year"].between(2022, 2024)]
cancel_val = cancel[cancel["Year"] == 2025]

for name, sample in [("Train", cancel_train), ("Validation", cancel_val)]:
    print(name, len(sample), "flights")
    print("Cancelled:", int(sample["Cancelled"].sum()))
    print("Cancellation rate:", round(sample["Cancelled"].mean() * 100, 2), "%")

Train 420483 flights
Cancelled: 9117
Cancellation rate: 2.17 %
Validation 142859 flights
Cancelled: 2395
Cancellation rate: 1.68 %


In [36]:
# 沿用延误模型的五个特征：航空公司、目的地、月份、星期、起飞小时
cancel_encoder = OneHotEncoder(handle_unknown="ignore")

X_cancel_train = cancel_encoder.fit_transform(cancel_train[features])
X_cancel_val = cancel_encoder.transform(cancel_val[features])

# 这次的正类是航班取消：1 = 取消，0 = 未取消
y_cancel_train = cancel_train["Cancelled"].astype(int)
y_cancel_val = cancel_val["Cancelled"].astype(int)

In [37]:
cancel_results = []

for weight in [None, "balanced"]:
    model = LogisticRegression(class_weight=weight, max_iter=500)
    model.fit(X_cancel_train, y_cancel_train)
    pred = model.predict(X_cancel_val)

    cancel_results.append({
        "Class weight": "Default" if weight is None else "Balanced",
        "Predicted cancellations": int(pred.sum()),
        "Accuracy": accuracy_score(y_cancel_val, pred),
        "Precision": precision_score(y_cancel_val, pred, zero_division=0),
        "Recall": recall_score(y_cancel_val, pred, zero_division=0),
        "F1": f1_score(y_cancel_val, pred, zero_division=0),
    })

pd.DataFrame(cancel_results).round(4)

,Class weight,Predicted cancellations,Accuracy,Precision,Recall,F1
0,Default,0,0.9832,0.000,0.0000,0.0000
1,Balanced,50613,0.6481,0.027,0.5704,0.0515


In [38]:
# 按取消任务的航班顺序取预报，让天气矩阵和时刻表矩阵逐行对应
cancel_taf_train = taf[taf_cols].reindex(cancel_train["FlightRowID"])
cancel_taf_val = taf[taf_cols].reindex(cancel_val["FlightRowID"])

# 补缺失值前，先看两个预报特征各自覆盖了多少航班
pd.DataFrame({
    "Train coverage": cancel_taf_train.notna().mean(),
    "Validation coverage": cancel_taf_val.notna().mean(),
}).round(3)

,Train coverage,Validation coverage
TAFWindKt_6h,0.999,0.999
TAFVisBelow3_6h,0.999,0.999


In [39]:
# 用取消任务训练集的中位数补缺失，保留全部验证航班
cancel_taf_median = cancel_taf_train.median()
train_weather = cancel_taf_train.fillna(cancel_taf_median)
val_weather = cancel_taf_val.fillna(cancel_taf_median)

# 从训练集学习标准化参数，再应用到验证集
cancel_taf_scaler = StandardScaler()
train_weather = cancel_taf_scaler.fit_transform(train_weather)
val_weather = cancel_taf_scaler.transform(val_weather)

X_cancel_train_6h = hstack([X_cancel_train, train_weather], format="csr")
X_cancel_val_6h = hstack([X_cancel_val, val_weather], format="csr")

In [40]:
cancel_experiments = [
    ("Schedule only", X_cancel_train, X_cancel_val),
    ("Schedule + 6h BOS TAF", X_cancel_train_6h, X_cancel_val_6h),
]

cancel_weather_results = []

for name, train_x, val_x in cancel_experiments:
    model = LogisticRegression(class_weight="balanced", max_iter=500)
    model.fit(train_x, y_cancel_train)
    pred = model.predict(val_x)

    cancel_weather_results.append({
        "Features": name,
        "Alerts": int(pred.sum()),
        "Cancellations caught": int(((pred == 1) & (y_cancel_val == 1)).sum()),
        "Precision": precision_score(y_cancel_val, pred, zero_division=0),
        "Recall": recall_score(y_cancel_val, pred, zero_division=0),
        "F1": f1_score(y_cancel_val, pred, zero_division=0),
    })

pd.DataFrame(cancel_weather_results).round(4)

,Features,Alerts,Cancellations caught,Precision,Recall,F1
0,Schedule only,50613,1366,0.027,0.5704,0.0515
1,Schedule + 6h BOS TAF,45111,1397,0.031,0.5833,0.0588


In [41]:
cancel_6h_model = LogisticRegression(class_weight="balanced", max_iter=500)
cancel_6h_model.fit(X_cancel_train_6h, y_cancel_train)

# 第 1 列对应标签 1，即“取消”的模型分数
cancel_scores = cancel_6h_model.predict_proba(X_cancel_val_6h)[:, 1]

In [42]:
threshold_results = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    # 模型只训练一次，每轮只改变发提醒的门槛
    pred = (cancel_scores > threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Alerts": int(pred.sum()),
        "Cancellations caught": int(((pred == 1) & (y_cancel_val == 1)).sum()),
        "Precision": precision_score(y_cancel_val, pred, zero_division=0),
        "Recall": recall_score(y_cancel_val, pred),
        "F1": f1_score(y_cancel_val, pred),
    })

threshold_table = pd.DataFrame(threshold_results)
threshold_table.round(4)

,Threshold,Alerts,Cancellations caught,Precision,Recall,F1
0,0.5,45111,1397,0.0310,0.5833,0.0588
1,0.6,25592,1029,0.0402,0.4296,0.0735
2,0.7,11390,620,0.0544,0.2589,0.0900
3,0.8,3344,285,0.0852,0.1190,0.0993
4,0.9,391,97,0.2481,0.0405,0.0696


In [43]:
# 从 2025 验证结果中选出 F1 最高的阈值
best_threshold = threshold_table.loc[threshold_table["F1"].idxmax(), "Threshold"]
print("Selected threshold:", best_threshold)

# 按航班编号接上两个 BOS 预报特征
cancel_data = cancel.join(taf[taf_cols], on="FlightRowID", validate="one_to_one")

cancel_final_train = cancel_data[cancel_data["Year"].between(2022, 2025)].copy()
cancel_test = cancel_data[cancel_data["Year"] == 2026].copy()

Selected threshold: 0.8


In [45]:
# 五个时刻表特征沿用前面的 features
cancel_final_encoder = OneHotEncoder(handle_unknown="ignore")
schedule_train = cancel_final_encoder.fit_transform(cancel_final_train[features])
schedule_test = cancel_final_encoder.transform(cancel_test[features])

# 两个天气特征沿用前面的 taf_cols
taf_medians = cancel_final_train[taf_cols].median()
taf_train = cancel_final_train[taf_cols].fillna(taf_medians)
taf_test = cancel_test[taf_cols].fillna(taf_medians)

cancel_final_scaler = StandardScaler()
taf_train = cancel_final_scaler.fit_transform(taf_train)
taf_test = cancel_final_scaler.transform(taf_test)

X_cancel_final = hstack([schedule_train, taf_train], format="csr")
X_cancel_test = hstack([schedule_test, taf_test], format="csr")

In [46]:
cancel_final_model = LogisticRegression(class_weight="balanced", max_iter=500)
cancel_final_model.fit(X_cancel_final, cancel_final_train["Cancelled"].astype(int))

y_cancel_test = cancel_test["Cancelled"].astype(int)
test_scores = cancel_final_model.predict_proba(X_cancel_test)[:, 1]

# 保持和 Cell 20 相同的规则：分数严格大于选定阈值才发提醒
cancel_test_pred = (test_scores > best_threshold).astype(int)

print("Test flights:", len(y_cancel_test))
print("Actual cancellations:", int(y_cancel_test.sum()))
print("Alerts:", int(cancel_test_pred.sum()))
print("Accuracy:", round(accuracy_score(y_cancel_test, cancel_test_pred), 4))
print("Precision:", round(precision_score(y_cancel_test, cancel_test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_cancel_test, cancel_test_pred), 4))
print("F1:", round(f1_score(y_cancel_test, cancel_test_pred), 4))

Test flights: 84109
Actual cancellations: 3064
Alerts: 3039
Accuracy: 0.9465
Precision: 0.2636
Recall: 0.2614
F1: 0.2625


In [47]:
pd.DataFrame(
    confusion_matrix(y_cancel_test, cancel_test_pred, labels=[0, 1]),
    index=["Actual: operated", "Actual: cancelled"],
    columns=["Predicted: operated", "Predicted: cancelled"],
)

,Predicted: operated,Predicted: cancelled
Actual: operated,78807,2238
Actual: cancelled,2263,801


Modeling Methods

We modeled arrival delays of at least 15 minutes and flight cancellations as separate binary classification tasks. We began with airline, destination, month, day of week, and scheduled departure hour because these features are available at booking. For arrival delays, we added historical BOS weather frequencies to represent typical conditions for the departure month and hour. For both tasks, we tested whether forecast wind speed and visibility available six hours before scheduled departure provided additional information.

We trained on 2022–2024 flights and used 2025 for model selection. We evaluated precision, recall, and F1 because cancellations were rare and accuracy alone rewarded models that predicted every flight would operate. For arrival delays, class-weighted logistic regression achieved a validation F1 of 0.4399, compared with 0.3875 for the best decision tree tested. For cancellations, we selected a threshold of 0.8 because it achieved the highest validation F1 among the five thresholds tested. We then retrained on 2022–2025 data and evaluated both models on January–July 2026 flights.

Results & Interpretation

For arrival delays, validation F1 increased from 0.4309 with schedule features alone to 0.4333 with historical weather and 0.4399 after adding the six-hour forecast. On the 2026 test set, the selected model identified 12,575 of 19,693 delayed flights, achieving 63.86% recall, 35.41% precision, and an F1 of 0.4556.

For cancellations, adding the forecast at a fixed threshold of 0.5 produced 5,502 fewer alerts while identifying 31 additional cancellations on the validation set. At the selected threshold of 0.8, the final model identified 801 of 3,064 cancellations on the test set, with 26.36% precision, 26.14% recall, and an F1 of 0.2625. Flagged flights had a cancellation rate about seven times the overall test rate, although the model missed most cancellations. The data record the final cancellation status but do not indicate when each cancellation was announced.